# Deep Learning for Conspiracy Detection (Features Only)
## Predicting conspiracy labels (yes/no/cant_tell) using Deep Neural Network

This notebook implements a deep learning approach using a Deep Neural Network (DNN) for:
1. **Conspiracy Label Prediction**: yes/no/cant_tell (multiclass classification)
2. **Model**: Deep Neural Network using only engineered features (no text embeddings)
3. **Features**: Uses engineered features only (lexical complexity, discourse markers, sentiment, POS counts, readability, etc.)
4. **Optimizer**: Adam optimizer
5. **Model Evaluation**: Training with validation split and evaluation on held-out test set (100 samples)

**Note**: This approach uses only the engineered linguistic features to predict conspiracy labels, without any text embeddings.

## Environment Setup

This notebook requires **NumPy 2.x** and compatible packages. The virtual environment should be set up using the `recreate_venv.sh` script which installs all packages with NumPy 2.x compatibility.


In [ ]:
# Verify environment setup and package versions
# Suppress tokenizers parallelism warning
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Import and verify all packages
import numpy as np
import pandas as pd
import torch
from transformers import __version__ as transformers_version

print("=" * 60)
print("Environment Verification")
print("=" * 60)

# Check NumPy
print(f"✓ NumPy version: {np.__version__}")
if not np.__version__.startswith('2.'):
    print("⚠ WARNING: NumPy 2.x is required but version", np.__version__, "is installed")

# Check PyTorch
print(f"✓ PyTorch version: {torch.__version__}")
try:
    test_tensor = torch.tensor([1, 2, 3])
    test_numpy = test_tensor.numpy()
    print("✓ PyTorch-NumPy compatibility: OK")
except Exception as e:
    print(f"❌ PyTorch-NumPy compatibility issue: {e}")
    print("   This may indicate PyTorch was not compiled with NumPy 2.x support")
    print("   Please run: bash EDA-Rehydrated/recreate_venv.sh")

# Check Transformers
print(f"✓ Transformers version: {transformers_version}")

# Check for GPU support (CUDA or ROCm)
is_rocm = hasattr(torch.version, 'hip') and torch.version.hip is not None
is_cuda_available = torch.cuda.is_available()

if is_rocm:
    print(f"✓ ROCm available: True")
    print(f"✓ ROCm version: {torch.version.hip}")
    print(f"✓ Device count: {torch.cuda.device_count()}")
    if torch.cuda.device_count() > 0:
        print(f"✓ Primary device: {torch.cuda.get_device_name(0)}")
        print(f"ℹ Training will use ROCm GPU(s)")
elif is_cuda_available:
    print(f"✓ CUDA available: True")
    print(f"✓ CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA version: {torch.version.cuda}")
    print(f"ℹ Training will use CUDA GPU(s)")
else:
    print("✓ CUDA/ROCm available: False")
    print("ℹ Training will use CPU")

print("=" * 60)
print("✓ Environment check complete")
print("=" * 60)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Ensure numpy is properly available for transformers
# This is a workaround for numpy 2.x compatibility issues
import sys
if 'numpy' not in sys.modules:
    import numpy
    sys.modules['numpy'] = numpy

# Deep learning libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset as TorchDataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Libraries imported successfully!")
print("=" * 50)
print("Environment Information:")
print("=" * 50)
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ NumPy version: {np.__version__}")

# Check for GPU support (CUDA or ROCm)
is_rocm = hasattr(torch.version, 'hip') and torch.version.hip is not None
is_cuda_available = torch.cuda.is_available()

if is_rocm:
    print(f"✓ ROCm available: True")
    print(f"✓ ROCm version: {torch.version.hip}")
    print(f"✓ Device count: {torch.cuda.device_count()}")
    if torch.cuda.device_count() > 0:
        print(f"✓ Primary device: {torch.cuda.get_device_name(0)}")
        print(f"ℹ Training will use ROCm GPU(s)")
elif is_cuda_available:
    print(f"✓ CUDA available: True")
    print(f"✓ CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA version: {torch.version.cuda}")
    print(f"ℹ Training will use CUDA GPU(s)")
else:
    print("✓ CUDA/ROCm available: False")
    print("⚠ Training will use CPU (slower than GPU)")
print("=" * 50)


## Load Data and Merge Features

Load the base data and merge engineered features from feature files. The model will use both text embeddings and engineered features.


In [ ]:
# Load base data and merge features (same approach as DecisionTree notebook)
BASE = Path('../')
PROC = BASE / 'data_processed'
FEAT = BASE / 'features'

# Load base data from data_clean.csv (contains _id, text, conspiracy)
df = pd.read_csv(PROC / 'data_clean.csv')

print(f"Base data: {df.shape}")
print(f"Base columns: {list(df.columns)}")
print(f"Base data row count: {len(df)}")

# Preserve original columns from data_clean.csv
base_cols = ['_id', 'text', 'conspiracy']
for col in base_cols:
    if col not in df.columns:
        print(f"WARNING: {col} not found in base data!")

# Store base _id values to ensure we only keep these rows
base_ids = set(df['_id'].values)
print(f"Unique _id values in base: {len(base_ids)}")

# Load feature files
feature_files = [
    FEAT / 'lexical_complexity.csv',
    FEAT / 'discourse_markers.csv',
    FEAT / 'sentiment_emotion.csv',
    FEAT / 'token_pos_counts.csv',
    FEAT / 'readability_scores.csv',
    FEAT / 'ma_ttr_mtld_scores.csv',
    FEAT / 'pos_analysis.csv'
]

# Start with base data - this preserves _id, text, conspiracy from data_clean.csv
merged = df.copy()
initial_row_count = len(merged)

# Merge feature files, ensuring we keep original columns from data_clean.csv
# and preserve the exact row count from base data
for feat_file in feature_files:
    if feat_file.exists():
        try:
            feat_df = pd.read_csv(feat_file)
            print(f"\nLoading {feat_file.name}: {feat_df.shape[0]} rows, {feat_df.shape[1]} columns")
            
            if '_id' in merged.columns and '_id' in feat_df.columns:
                # Filter feature file to only include rows with _id in base data
                feat_df_filtered = feat_df[feat_df['_id'].isin(base_ids)].copy()
                print(f"  Filtered to {len(feat_df_filtered)} rows matching base _id values")
                
                # Remove duplicates by _id (keep first occurrence) to prevent row explosion
                initial_filtered_count = len(feat_df_filtered)
                feat_df_filtered = feat_df_filtered.drop_duplicates(subset=['_id'], keep='first')
                if len(feat_df_filtered) < initial_filtered_count:
                    print(f"  Removed {initial_filtered_count - len(feat_df_filtered)} duplicate _id rows (kept first)")
                
                # Drop any columns from feature file that conflict with base_cols (except _id)
                cols_to_drop = [col for col in feat_df_filtered.columns if col in base_cols and col != '_id']
                if cols_to_drop:
                    feat_df_filtered = feat_df_filtered.drop(columns=cols_to_drop)
                    print(f"  Dropped conflicting columns: {cols_to_drop}")
                
                # Merge on _id, keeping all rows from merged (left join)
                merged = merged.merge(feat_df_filtered, on='_id', how='left')
                print(f"  Merged successfully. Merged shape: {merged.shape}")
            else:
                print(f"  Skipped: missing '_id' column")
        except Exception as e:
            print(f"Could not load {feat_file.name}: {e}")

# Verify that original columns from data_clean.csv are still present
for col in base_cols:
    if col not in merged.columns:
        print(f"ERROR: {col} was lost during merge!")

# Verify row count is preserved
if len(merged) != initial_row_count:
    print(f"\nWARNING: Row count changed from {initial_row_count} to {len(merged)}!")
else:
    print(f"\n✓ Row count preserved: {initial_row_count}")

print(f"\nFinal merged shape: {merged.shape}")
print(f"Total columns: {len(merged.columns)}")
print(f"Original columns preserved: {all(col in merged.columns for col in base_cols)}")

# Remove any rows with missing text or labels
df_model = merged.dropna(subset=['text', 'conspiracy'])
print(f"\nAfter removing missing values: {len(df_model)} documents")
print(f"\nLabel distribution after cleaning:")
print(df_model['conspiracy'].value_counts())

# Identify feature columns (all columns except base_cols)
feature_columns = [col for col in df_model.columns if col not in base_cols]
print(f"\n✓ Feature columns identified: {len(feature_columns)} features")
print(f"  First 10 feature columns: {feature_columns[:10]}")

# Display sample texts
print("\n=== Sample Texts ===")
for label in ['yes', 'no', 'cant_tell']:
    sample = df_model[df_model['conspiracy'] == label].iloc[0]
    print(f"\nLabel: {label}")
    print(f"Text (first 200 chars): {sample['text'][:200]}...")


## Create Train/Test Split

First, split the data into training set and test set (100 samples held out for final evaluation). The training set will later be split into train/validation for model training.


In [ ]:
# Load test set IDs (same as other notebooks - these are held out for final evaluation)
TEST_FILE = PROC / 'validation_set_ids.csv'  # Note: file is named validation_set_ids but we use it as test set

if TEST_FILE.exists():
    print("Loading existing test set...")
    test_ids_df = pd.read_csv(TEST_FILE)
    test_ids = set(test_ids_df['_id'].values)
    print(f"  Loaded {len(test_ids)} test IDs from {TEST_FILE.name}")
else:
    print("WARNING: Test set file not found. Creating new test set...")
    # Create test set using stratified sampling
    TEST_SIZE = 100
    target_fraction = TEST_SIZE / len(df_model)
    
    train_ids, test_ids_list = train_test_split(
        df_model['_id'].values,
        test_size=target_fraction,
        stratify=df_model['conspiracy'].values,
        random_state=42
    )
    test_ids = set(test_ids_list[:TEST_SIZE])
    
    # Save test set IDs
    test_ids_df = pd.DataFrame({'_id': list(test_ids)})
    test_ids_df.to_csv(TEST_FILE, index=False)
    print(f"  ✓ Saved test set to {TEST_FILE.name}")

# Split data using test IDs
# Use the merged dataframe with features
df_with_id = df_model.copy()  # df_model already has all features merged

# Test set: held out for final evaluation
test_df = df_with_id[df_with_id['_id'].isin(test_ids)].copy()
# Training set: will be split into train/validation for model training
train_df = df_with_id[~df_with_id['_id'].isin(test_ids)].copy()

print(f"\nTraining set size: {len(train_df)} samples (will be split into train/validation)")
print(f"Training set class distribution:")
print(train_df['conspiracy'].value_counts().to_dict())

print(f"\nTest set size: {len(test_df)} samples (held out for final evaluation)")
print(f"Test set class distribution:")
print(test_df['conspiracy'].value_counts().to_dict())

# Store feature column names (will be used later)
feature_columns = [col for col in train_df.columns if col not in ['_id', 'text', 'conspiracy']]
print(f"\n✓ Feature columns to be used: {len(feature_columns)} features")
print(f"  Sample features: {feature_columns[:5]}")


In [ ]:
# Model configuration
VALIDATION_SPLIT = 0.2  # 20% for validation, 80% for training

# Encode labels (fit on training set only)
label_encoder = LabelEncoder()
label_encoder.fit(train_df['conspiracy'])

# Create label mappings
label_to_id = {label: int(id) for label, id in zip(label_encoder.classes_, range(len(label_encoder.classes_)))}
id_to_label = {int(id): label for label, id in label_to_id.items()}

print(f"Label mappings:")
print(f"  Label to ID: {label_to_id}")
print(f"  ID to Label: {id_to_label}")
print(f"  Number of classes: {len(label_to_id)}")

# Prepare labels for train/validation split
train_labels = label_encoder.transform(train_df['conspiracy']).tolist()

# Prepare test set labels (will be used only at the end)
test_labels = label_encoder.transform(test_df['conspiracy']).tolist()

print(f"\nTraining set: {len(train_df)} samples (will be split into train/validation)")
print(f"Test set: {len(test_df)} samples (held out for final evaluation)")
print(f"Number of features: {len(feature_columns)}")

print("\n✓ Data preparation complete!")
print("Using features only (no text processing needed)")


## Prepare Data for Training

Prepare the data for model training by encoding labels and preparing text lists. The training set will be split into train/validation for model training.


## Train/Validation Split Setup

Set up a stratified train/validation split on the training set.


In [ ]:
# Set up train/validation split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create stratified train/validation split
train_indices, val_indices = train_test_split(
    range(len(train_df)),
    test_size=VALIDATION_SPLIT,
    stratify=train_labels,
    random_state=42
)

print(f"Train/Validation Split Setup")
print("=" * 60)
print(f"Training samples: {len(train_indices)} ({100*(1-VALIDATION_SPLIT):.1f}%)")
print(f"Validation samples: {len(val_indices)} ({100*VALIDATION_SPLIT:.1f}%)")

# Show class distribution in validation set
val_labels_split = [train_labels[i] for i in val_indices]
val_label_counts = pd.Series(val_labels_split).value_counts().sort_index()
print(f"\nValidation set class distribution:")
print(dict(zip([id_to_label[i] for i in val_label_counts.index], val_label_counts.values)))

# Show class distribution in training set
train_labels_split = [train_labels[i] for i in train_indices]
train_label_counts = pd.Series(train_labels_split).value_counts().sort_index()
print(f"\nTraining set class distribution:")
print(dict(zip([id_to_label[i] for i in train_label_counts.index], train_label_counts.values)))
print("=" * 60)

# Prepare feature arrays for train/validation split
# Get feature arrays for train and validation splits
train_split_features = train_df.iloc[train_indices][feature_columns].fillna(0).values.astype(np.float32)
val_split_features = train_df.iloc[val_indices][feature_columns].fillna(0).values.astype(np.float32)

# Normalize features
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_split_features)
validation_features_scaled = scaler.transform(val_split_features)

# Also prepare test set features for final evaluation
test_features = test_df[feature_columns].fillna(0).values.astype(np.float32)
test_features_scaled = scaler.transform(test_features)

print(f"\n✓ Features prepared:")
print(f"  Number of features: {len(feature_columns)}")
print(f"  Training features shape: {train_features_scaled.shape}")
print(f"  Validation features shape: {validation_features_scaled.shape}")
print(f"  Test features shape: {test_features_scaled.shape}")

# Get train/validation labels
train_split_labels = np.array([train_labels[i] for i in train_indices], dtype=np.int64)
val_split_labels = np.array([train_labels[i] for i in val_indices], dtype=np.int64)

# Convert to PyTorch tensors
train_features_tensor = torch.FloatTensor(train_features_scaled)
train_labels_tensor = torch.LongTensor(train_split_labels)
val_features_tensor = torch.FloatTensor(validation_features_scaled)
val_labels_tensor = torch.LongTensor(val_split_labels)

# Create PyTorch datasets
train_dataset = TensorDataset(train_features_tensor, train_labels_tensor)
validation_dataset = TensorDataset(val_features_tensor, val_labels_tensor)

print(f"\n✓ PyTorch datasets created:")
print(f"  Training dataset: {len(train_dataset)} samples")
print(f"  Validation dataset: {len(validation_dataset)} samples")
print(f"  Features per sample: {len(feature_columns)}")


In [ ]:
# This cell has been moved to Cell 7 (Prepare Data for Training)
# All data preparation (train_texts, train_labels, tokenizer, etc.) is now done in Cell 7
# This cell is kept empty to maintain notebook structure
pass


## Create Deep Neural Network Model

Create a deep neural network that uses only engineered features to predict conspiracy labels.


In [ ]:
# Create Deep Neural Network model
class FeatureBasedDNN(nn.Module):
    """Deep Neural Network with residual connections and skip layers"""
    
    def __init__(self, input_dim, num_classes, hidden_dims=[512, 512, 256, 256, 128, 128, 64], dropout_rate=0.4):
        super(FeatureBasedDNN, self).__init__()
        
        self.input_dim = input_dim
        self.num_classes = num_classes
        
        # Input projection with expansion
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]),
            nn.BatchNorm1d(hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5)
        )
        
        # Build deep hidden layers with residual connections
        self.layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()
        self.dropout_layers = nn.ModuleList()
        
        for i in range(len(hidden_dims) - 1):
            # Main layer
            self.layers.append(nn.Linear(hidden_dims[i], hidden_dims[i+1]))
            self.bn_layers.append(nn.BatchNorm1d(hidden_dims[i+1]))
            self.dropout_layers.append(nn.Dropout(dropout_rate))
            
            # Mark if residual connection is possible
            if hidden_dims[i] == hidden_dims[i+1]:
                self.layers[-1].residual = True
            else:
                self.layers[-1].residual = False
        
        # Additional deep layers with skip connections
        self.skip_layers = nn.ModuleList()
        for i in range(len(hidden_dims) - 2):
            if hidden_dims[i] == hidden_dims[i+2]:
                self.skip_layers.append(nn.Linear(hidden_dims[i], hidden_dims[i+2]))
            else:
                self.skip_layers.append(None)
        
        # Output layer with multiple paths
        self.output_proj1 = nn.Sequential(
            nn.Linear(hidden_dims[-1], hidden_dims[-1] // 2),
            nn.BatchNorm1d(hidden_dims[-1] // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5)
        )
        
        self.output_proj2 = nn.Sequential(
            nn.Linear(hidden_dims[-1] // 2, hidden_dims[-1] // 4),
            nn.BatchNorm1d(hidden_dims[-1] // 4),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.3)
        )
        
        self.classifier = nn.Linear(hidden_dims[-1] // 4, num_classes)
        
    def forward(self, x):
        # Input projection
        out = self.input_proj(x)
        prev_out = out
        
        # Deep layers with residual connections
        for i, (layer, bn, dropout) in enumerate(zip(self.layers, self.bn_layers, self.dropout_layers)):
            residual = out
            
            # Forward pass
            out = layer(out)
            out = bn(out)
            out = nn.ReLU()(out)
            out = dropout(out)
            
            # Residual connection (if dimensions match)
            if hasattr(layer, 'residual') and layer.residual:
                out = out + residual
            
            # Skip connection (every other layer)
            if i >= 2 and i % 2 == 0 and self.skip_layers[i-2] is not None:
                skip = self.skip_layers[i-2](prev_out)
                out = out + skip
                prev_out = out
        
        # Output projection
        out = self.output_proj1(out)
        out = self.output_proj2(out)
        out = self.classifier(out)
        
        return out

# Model configuration
num_features = len(feature_columns)
num_labels = len(label_to_id)

# Create model
# Create deeper, more complex model
model = FeatureBasedDNN(
    input_dim=num_features,
    num_classes=num_labels,
    hidden_dims=[512, 512, 256, 256, 128, 128, 64],  # Much deeper network
    dropout_rate=0.4  # Slightly higher dropout for regularization
)

print(f"Creating Deep Neural Network:")
print(f"  Input features: {num_features}")
print(f"  Number of classes: {num_labels}")
print(f"  Hidden layers: [512, 512, 256, 256, 128, 128, 64] (7 layers)")
print(f"  Dropout rate: 0.4")
print(f"  Architecture: Residual connections + Skip connections")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Training hyperparameters
# Training hyperparameters - adjusted for deeper model
BATCH_SIZE = 32  # Smaller batch for deeper model
LEARNING_RATE = 0.0005  # Lower learning rate for stability
NUM_EPOCHS = 100  # More epochs for deeper model
WEIGHT_DECAY = 0.0005  # Higher weight decay for regularization

print(f"\nTraining Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Optimizer: Adam")
print(f"  Using features only: Yes ({num_features} features)")

# Setup device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n✓ Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Move model to device
model = model.to(device)
print(f"✓ Model moved to {device}")

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n✓ Data loaders created:")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

# Define loss function with class weights (handle imbalance)
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights to handle imbalance
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Use AdamW optimizer (better weight decay) with learning rate scheduler
optimizer = optim.AdamW(
    model.parameters(), 
    lr=LEARNING_RATE, 
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8
)

# Learning rate scheduler (reduce on plateau)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max',  # Monitor F1-macro (higher is better)
    factor=0.5,  # Reduce LR by half
    patience=5,  # Wait 5 epochs before reducing
    verbose=True,
    min_lr=1e-6
)

print(f"\n✓ Optimizer: AdamW (with weight decay)")
print(f"✓ Loss function: CrossEntropyLoss (with class weights for imbalance)")
print(f"✓ Learning rate scheduler: ReduceLROnPlateau")
print(f"✓ Class weights: {dict(zip([id_to_label[i] for i in range(len(class_weights))], class_weights.round(3)))}")


In [ ]:
# Training and evaluation functions
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def evaluate(model, val_loader, criterion, device):
    """Evaluate on validation set"""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for features, labels in val_loader:
            features = features.to(device)
            labels = labels.to(device)
            
            outputs = model(features)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(val_loader)
    accuracy = accuracy_score(all_labels, all_predictions)
    f1_macro = f1_score(all_labels, all_predictions, average='macro')
    f1_weighted = f1_score(all_labels, all_predictions, average='weighted')
    
    return avg_loss, accuracy, f1_macro, f1_weighted, all_predictions, all_labels

print("✓ Training and evaluation functions defined")


## Train the Model

Train the Deep Neural Network using Adam optimizer on features only.


In [ ]:
# Training loop
print("Starting training...")
print("=" * 60)
print(f"Training on {len(train_dataset)} samples")
print(f"Validation on {len(validation_dataset)} samples")
print(f"Model: Deep Neural Network (Features Only)")
print(f"Epochs: {NUM_EPOCHS}, Batch size: {BATCH_SIZE}, Learning rate: {LEARNING_RATE}")
print(f"Device: {device}")
print("=" * 60)

# Training history
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_macros = []
val_f1_weighteds = []

best_val_f1 = 0
best_model_state = None
patience = 15  # Increased patience for deeper model
patience_counter = 0
current_lr = LEARNING_RATE

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc, val_f1_macro, val_f1_weighted, _, _ = evaluate(model, val_loader, criterion, device)
    
    # Update learning rate scheduler
    scheduler.step(val_f1_macro)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Store history
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_f1_macros.append(val_f1_macro)
    val_f1_weighteds.append(val_f1_weighted)
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1-macro: {val_f1_macro:.4f}")
        print(f"  Learning Rate: {current_lr:.6f}")
    
    # Early stopping based on F1-macro
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        print(f"  ✓ New best F1-macro: {best_val_f1:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            print(f"Best validation F1-macro: {best_val_f1:.4f}")
            break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\n✓ Loaded best model (Val F1-macro: {best_val_f1:.4f})")

print("\n" + "=" * 60)
print("✓ Training complete!")
print("=" * 60)


## Plot Training History

Visualize training and validation metrics over epochs.


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Loss
axes[0].plot(train_losses, label='Train Loss', marker='o')
axes[0].plot(val_losses, label='Validation Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Plot 2: Accuracy and F1
axes[1].plot(train_accuracies, label='Train Accuracy', marker='o')
axes[1].plot(val_accuracies, label='Validation Accuracy', marker='s')
axes[1].plot([f1 * 100 for f1 in val_f1_macros], label='Validation F1-macro', marker='^')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score (%)')
axes[1].set_title('Training and Validation Metrics')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()

# Save figure
figures_dir = BASE / 'figures'
figures_dir.mkdir(exist_ok=True)
plt.savefig(figures_dir / 'dnn_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Training history plot saved to figures/dnn_training_history.png")
print(f"\nFinal Training Metrics:")
print(f"  Best Validation F1-macro: {best_val_f1:.4f}")
print(f"  Final Validation Accuracy: {val_accuracies[-1]:.4f}")
print(f"  Final Validation F1-weighted: {val_f1_weighteds[-1]:.4f}")


## Evaluate on Validation Set


In [ ]:
# Evaluate on validation set
print("Evaluating on validation set...")
print("=" * 60)
print("✓ Using Deep Neural Network with engineered features only")
print(f"✓ Validation dataset contains {len(feature_columns)} features per sample")

val_loss, val_accuracy, val_f1_macro, val_f1_weighted, val_predictions, val_true_labels = evaluate(
    model, val_loader, criterion, device
)

print("\n=== Validation Set Performance ===")
print("=" * 60)
print(f"  Loss: {val_loss:.4f}")
print(f"  Accuracy: {val_accuracy:.4f}")
print(f"  F1-macro: {val_f1_macro:.4f}")
print(f"  F1-weighted: {val_f1_weighted:.4f}")
print("=" * 60)
print("✓ Evaluation complete - predictions used engineered features only")


In [ ]:
# Get predictions for detailed analysis
print("\n=== Validation Set Predictions ===")

# Convert to label names
y_true_labels = [id_to_label[label] for label in val_true_labels]
y_pred_labels = [id_to_label[label] for label in val_predictions]
true_labels = val_true_labels
predicted_classes = val_predictions

print("=== Validation Set Predictions ===")
print(f"\nTrue label distribution:")
print(pd.Series(y_true_labels).value_counts().to_dict())

print(f"\nPredicted label distribution:")
print(pd.Series(y_pred_labels).value_counts().to_dict())

# Calculate metrics
val_accuracy = accuracy_score(true_labels, predicted_classes)
val_f1_macro = f1_score(true_labels, predicted_classes, average='macro')
val_f1_weighted = f1_score(true_labels, predicted_classes, average='weighted')

print(f"\n=== Validation Set Performance (Detailed) ===")
print(f"Accuracy: {val_accuracy:.4f}")
print(f"F1 (macro): {val_f1_macro:.4f}")
print(f"F1 (weighted): {val_f1_weighted:.4f}")

# Detailed classification report
print(f"\n=== Detailed Classification Report ===")
print(classification_report(y_true_labels, y_pred_labels, target_names=label_encoder.classes_))


In [ ]:
# Confusion matrix
cm = confusion_matrix(true_labels, predicted_classes)
print(f"\n=== Confusion Matrix ===")
print("Rows = True labels, Columns = Predicted labels")
print(f"Label order: {list(label_encoder.classes_)}")
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Validation Set (Deep Neural Network - Features Only)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()

# Save figure
figures_dir = BASE / 'figures'
figures_dir.mkdir(exist_ok=True)
plt.savefig(figures_dir / 'validation_confusion_matrix_dnn_features.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ Saved confusion matrix to figures/validation_confusion_matrix_dnn_features.png")


## Sample Predictions


## Final Evaluation on Test Set

Evaluate the best model from training on the held-out test set (100 samples).


In [ ]:
# Compare predictions with true labels for some examples
print("\n=== Sample Validation Set Predictions ===")
sample_indices = np.random.choice(len(val_split_texts), size=min(10, len(val_split_texts)), replace=False)

results_df = pd.DataFrame({
    'true_label': [y_true_labels[i] for i in sample_indices],
    'predicted_label': [y_pred_labels[i] for i in sample_indices],
    'text_preview': [val_split_texts[i][:100] + '...' for i in sample_indices],
    'correct': [y_true_labels[i] == y_pred_labels[i] for i in sample_indices]
})

print(results_df.to_string(index=False))

print(f"\n✓ Validation set evaluation complete!")
print(f"  Total validation samples: {len(val_split_texts)}")
print(f"  Accuracy: {val_accuracy:.4f}")
print(f"  Correct predictions: {(np.array(y_true_labels) == np.array(y_pred_labels)).sum()}/{len(y_true_labels)}")


In [ ]:
# Prepare test set for evaluation
print("Preparing test set for final evaluation...")
print("=" * 60)

# Verify that test_df has all required features
print("\n✓ Verifying features in test set...")
missing_features = [col for col in feature_columns if col not in test_df.columns]
if missing_features:
    print(f"  ⚠ Warning: {len(missing_features)} features missing in test_df")
    print(f"  Missing features: {missing_features[:10]}...")
    print(f"  Computing missing features by loading feature files...")
    
    # Load feature files and merge with test_df
    BASE = Path('../')
    FEAT = BASE / 'features'
    
    feature_files = [
        FEAT / 'lexical_complexity.csv',
        FEAT / 'discourse_markers.csv',
        FEAT / 'sentiment_emotion.csv',
        FEAT / 'token_pos_counts.csv',
        FEAT / 'readability_scores.csv',
        FEAT / 'ma_ttr_mtld_scores.csv',
        FEAT / 'pos_analysis.csv'
    ]
    
    test_ids = set(test_df['_id'].values)
    test_df_with_features = test_df.copy()
    
    for feat_file in feature_files:
        if feat_file.exists():
            try:
                feat_df = pd.read_csv(feat_file)
                if '_id' in feat_df.columns:
                    feat_df_filtered = feat_df[feat_df['_id'].isin(test_ids)].copy()
                    feat_df_filtered = feat_df_filtered.drop_duplicates(subset=['_id'], keep='first')
                    
                    # Drop conflicting columns (except _id)
                    cols_to_drop = [col for col in feat_df_filtered.columns if col in ['_id', 'text', 'conspiracy'] and col != '_id']
                    if cols_to_drop:
                        feat_df_filtered = feat_df_filtered.drop(columns=cols_to_drop)
                    
                    test_df_with_features = test_df_with_features.merge(feat_df_filtered, on='_id', how='left')
                    print(f"    ✓ Merged {feat_file.name}")
            except Exception as e:
                print(f"    ⚠ Could not load {feat_file.name}: {e}")
    
    # Update test_df
    test_df = test_df_with_features.copy()
    print(f"  ✓ Test set updated with features")
    
    # Re-extract features
    missing_features_after = [col for col in feature_columns if col not in test_df.columns]
    if missing_features_after:
        print(f"  ⚠ Still missing {len(missing_features_after)} features after merge")
        print(f"  Will fill with 0 for missing features")
    else:
        print(f"  ✓ All {len(feature_columns)} features now available in test set")
else:
    print(f"  ✓ All {len(feature_columns)} features already present in test_df")

# Extract and scale features for test set
print(f"\n✓ Extracting and scaling features for test set...")
test_features = test_df[feature_columns].fillna(0).values.astype(np.float32)
test_features_scaled = scaler.transform(test_features)
print(f"  Test features shape: {test_features_scaled.shape}")
print(f"  ✓ Features scaled using the same scaler from training")

# Convert to PyTorch tensors
test_features_tensor = torch.FloatTensor(test_features_scaled)
test_labels_tensor = torch.LongTensor(test_labels)

# Create PyTorch dataset and dataloader
test_dataset = TensorDataset(test_features_tensor, test_labels_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n✓ Test set prepared: {len(test_dataset)} samples")
print(f"  Features per sample: {len(feature_columns)}")
print("=" * 60)

# Evaluate on test set
print("\n" + "=" * 60)
print("Evaluating on Test Set (100 held-out samples)")
print("=" * 60)
print("✓ Using Deep Neural Network with engineered features only")

test_loss, test_accuracy, test_f1_macro, test_f1_weighted, test_predictions, test_true_labels = evaluate(
    model, test_loader, criterion, device
)

print("\n=== Test Set Performance ===")
print("=" * 60)
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f}")
print(f"  F1-macro: {test_f1_macro:.4f}")
print(f"  F1-weighted: {test_f1_weighted:.4f}")
print("=" * 60)
print("✓ Evaluation complete - predictions used engineered features only")


In [ ]:
# Get predictions for detailed analysis
print("\n=== Test Set Predictions ===")

# Use predictions from evaluation
predicted_classes = test_predictions
true_labels_test = test_true_labels

# Convert to label names
y_true_labels_test = [id_to_label[label] for label in true_labels_test]
y_pred_labels_test = [id_to_label[label] for label in predicted_classes]

print("\n=== Test Set Predictions ===")
print(f"\nTrue label distribution:")
print(pd.Series(y_true_labels_test).value_counts().to_dict())

print(f"\nPredicted label distribution:")
print(pd.Series(y_pred_labels_test).value_counts().to_dict())

# Calculate metrics
test_accuracy = accuracy_score(true_labels_test, predicted_classes)
test_f1_macro = f1_score(true_labels_test, predicted_classes, average='macro')
test_f1_weighted = f1_score(true_labels_test, predicted_classes, average='weighted')

print(f"\n=== Test Set Performance (Detailed) ===")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"F1 (macro): {test_f1_macro:.4f}")
print(f"F1 (weighted): {test_f1_weighted:.4f}")

# Detailed classification report
print(f"\n=== Detailed Classification Report (Test Set) ===")
print(classification_report(y_true_labels_test, y_pred_labels_test, target_names=label_encoder.classes_))


In [ ]:
# Confusion matrix for test set
cm_test = confusion_matrix(true_labels_test, predicted_classes)
print(f"\n=== Confusion Matrix (Test Set) ===")
print("Rows = True labels, Columns = Predicted labels")
print(f"Label order: {list(label_encoder.classes_)}")
print(cm_test)

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Test Set (Deep Neural Network - Features Only)', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()

# Save figure
figures_dir = BASE / 'figures'
figures_dir.mkdir(exist_ok=True)
plt.savefig(figures_dir / 'test_confusion_matrix_dnn_features.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ Saved confusion matrix to figures/test_confusion_matrix_dnn_features.png")


## Sample Test Set Predictions


In [ ]:
# Compare predictions with true labels for some examples
print("\n=== Sample Test Set Predictions ===")
sample_indices = np.random.choice(len(test_df), size=min(10, len(test_df)), replace=False)

results_df = pd.DataFrame({
    'true_label': [y_true_labels_test[i] for i in sample_indices],
    'predicted_label': [y_pred_labels_test[i] for i in sample_indices],
    'text_preview': [test_df.iloc[i]['text'][:100] + '...' for i in sample_indices],
    'correct': [y_true_labels_test[i] == y_pred_labels_test[i] for i in sample_indices]
})

print(results_df.to_string(index=False))

print(f"\n✓ Test set evaluation complete!")
print(f"  Total test samples: {len(test_df)}")
print(f"  Accuracy: {test_accuracy:.4f}")
print(f"  F1 (macro): {test_f1_macro:.4f}")
print(f"  F1 (weighted): {test_f1_weighted:.4f}")
print(f"  Correct predictions: {(np.array(y_true_labels_test) == np.array(y_pred_labels_test)).sum()}/{len(y_true_labels_test)}")


This deep learning approach:
- **Uses engineered features only**: Uses 62+ engineered linguistic features (no text embeddings)
- **Model**: Deep Neural Network (DNN) with multiple hidden layers
- **Features Used**: 
  - Lexical complexity (TTR, hapax ratio, etc.)
  - Discourse markers
  - Sentiment and emotion scores
  - POS tag counts
  - Readability scores
  - MA-TTR and MTLD scores
  - POS analysis features
- **Evaluation Strategy**:
  - **Train/Validation Split**: 80% training, 20% validation (stratified split)
  - **Final Test Set**: 100 held-out samples used only for final evaluation
  - **Metrics**: Accuracy, F1-macro, F1-weighted reported for validation and test set
- **Optimizer**: Adam optimizer with weight decay
- **Advantages**: 
  - Fast training and inference (no transformer overhead)
  - Focuses purely on engineered linguistic features
  - Can capture patterns in lexical, syntactic, and discourse features
- **Architecture**: 
  - Input: 62 engineered features
  - Hidden layers: [256, 128, 64] with BatchNorm, ReLU, and Dropout
  - Output: 3 classes (yes/no/cant_tell)

**Note**: For production use, you might want to:
- Experiment with different transformer models (BERT, RoBERTa, etc.)
- Tune hyperparameters (learning rate, batch size, epochs)
- Use early stopping based on validation performance
- Apply class weighting if there's class imbalance
- Experiment with different feature combinations or feature selection


## Summary

This deep learning approach:
- **Uses engineered features only**: Uses 62+ engineered linguistic features (no text embeddings)
- **Model**: Deep Neural Network (DNN) with multiple hidden layers
- **Features Used**: 
  - Lexical complexity (TTR, hapax ratio, etc.)
  - Discourse markers
  - Sentiment and emotion scores
  - POS tag counts
  - Readability scores
  - MA-TTR and MTLD scores
  - POS analysis features
- **Evaluation Strategy**:
  - **Train/Validation Split**: 80% training, 20% validation (stratified split)
  - **Final Test Set**: 100 held-out samples used only for final evaluation
  - **Metrics**: Accuracy, F1-macro, F1-weighted reported for validation and test set
- **Optimizer**: Adam optimizer with weight decay
- **Advantages**: 
  - Fast training and inference (no transformer overhead)
  - Focuses purely on engineered linguistic features
  - Can capture patterns in lexical, syntactic, and discourse features
- **Architecture**: 
  - Input: 62 engineered features
  - Hidden layers: [256, 128, 64] with BatchNorm, ReLU, and Dropout
  - Output: 3 classes (yes/no/cant_tell)

**Note**: For production use, you might want to:
- Tune hyperparameters (learning rate, batch size, epochs, hidden layer sizes)
- Experiment with different network architectures
- Use early stopping based on validation performance
- Apply class weighting if there's class imbalance
- Experiment with different feature combinations or feature selection
